# Инспектор конвертации Qwen -> D16Qwen и визуализация орбит $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$

Этот блокнот визуализирует процесс сборки нативной модели `D16QwenForCausalLM` (`src/d16_qwen_model.py`) из весов вещественной сети Qwen с помощью конвертера `QwenToD16Converter` (`src/d16_qwen_converter.py`) и утилит `src/d16_vis_utils.py`:
1. **Обнаружение связанных четвёрок нейронов**, образующих единое комплексное число `D16Complex`.
2. **Полярный 16-секторный компас** (`[0°, 22.5°]` фундаментальный клин и секторы `S0..S15`) для отдельных четвёрок нейронов.
3. **Спектр всех 16 операций диэдральной группы $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$** (осевые `0..7` $\mathrm{Im}(z^2)$ против $45^\circ$-поворотов Адамара `8..15` $\mathrm{Re}(z^2)$) по слоям сети.
4. **Полярные профили 18-секторных таблиц усиления** `sector_im_gain` и `sector_re_gain`.

In [ ]:
%matplotlib inline
import json
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Подключаем папку src проекта D:\repo\d16nn\qwen\src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from d16_complex import D16Complex, D16ComplexTensor, STATE_LABELS
from d16_qwen_model import D16QwenConfig, D16QwenForCausalLM
from d16_qwen_converter import (
    OP_NAMES,
    QwenToD16Converter,
    convert_mlp_layer_to_d16,
    d16_group_matrices,
)
from d16_vis_utils import (
    plot_layer_z8z2_spectrum,
    plot_learned_sector_gains,
    plot_single_orbit_compass,
    print_orbit_table,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Using device: {device}")

## 1. Загрузка весов Qwen2.5-0.5B (или генерация эталонного чекпоинта с известными орбитами)
Если локальный чекпоинт `Qwen/Qwen2.5-0.5B` найден в кэше HuggingFace (или по пути `QWEN_LOCAL_PATH`), конвертируем реальные веса Qwen. Если локальный чекпоинт не указан, автоматически создаётся многослойный стенд с вшитыми орбитами $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$, чтобы все графики работали автономно.

In [ ]:
# Укажи путь к локальной папке с config.json и model.safetensors (или оставь None для автопоиска в HF cache)
QWEN_LOCAL_PATH = None
CONVERSION_MODE = "orbit_4to1"  # "orbit_4to1" (4 нейрона -> 1 D16Complex) или "full_1to1"

def find_local_qwen_snapshot(model_id: str = "Qwen/Qwen2.5-0.5B") -> Path | None:
    if QWEN_LOCAL_PATH and Path(QWEN_LOCAL_PATH).is_dir():
        return Path(QWEN_LOCAL_PATH)
    hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    hub_dir = hf_home / "hub" / ("models--" + model_id.replace("/", "--"))
    snaps = list((hub_dir / "snapshots").glob("*")) if (hub_dir / "snapshots").is_dir() else []
    for s in snaps:
        if (s / "config.json").is_file():
            return s
    return None


def build_synthetic_qwen_with_orbits(num_layers: int = 6, hidden_size: int = 64, inter_size: int = 128):
    """Создает стендовый state_dict Qwen с реалистичными орбитами Z_8 |>< Z_2 в каждом слое."""
    cfg = {
        "vocab_size": 256,
        "hidden_size": hidden_size,
        "intermediate_size": inter_size,
        "num_hidden_layers": num_layers,
        "num_attention_heads": 4,
        "num_key_value_heads": 2,
        "max_position_embeddings": 512,
        "rms_norm_eps": 1e-6,
        "rope_theta": 10000.0,
        "tie_word_embeddings": True,
    }
    mats = d16_group_matrices(dtype=torch.float32)
    kv_dim = (hidden_size // 4) * 2
    sd = {
        "model.embed_tokens.weight": torch.randn(256, hidden_size) * 0.02,
        "model.norm.weight": torch.ones(hidden_size),
    }
    K = inter_size // 4
    for l_idx in range(num_layers):
        p = f"model.layers.{l_idx}."
        sd[p + "input_layernorm.weight"] = torch.ones(hidden_size)
        sd[p + "post_attention_layernorm.weight"] = torch.ones(hidden_size)
        for proj, out_d in [("q_proj", hidden_size), ("k_proj", kv_dim), ("v_proj", kv_dim)]:
            sd[p + f"self_attn.{proj}.weight"] = torch.randn(out_d, hidden_size) * 0.02
            sd[p + f"self_attn.{proj}.bias"] = torch.zeros(out_d)
        sd[p + "self_attn.o_proj.weight"] = torch.randn(hidden_size, hidden_size) * 0.02

        base_B = F.normalize(torch.randn(K, 2, hidden_size), dim=-1) * 1.2
        U_rows, G_rows = [], []
        for q in range(K):
            # В средних слоях повышаем долю 45°-поворотов Адамара (op in 8..15)
            ops_q = [0, 1 + (q % 7), 8 + ((q + l_idx) % 8), 8 + ((q * 3 + l_idx) % 8)]
            for op_id in ops_q:
                noise = torch.randn(2, hidden_size) * (0.04 + 0.02 * (q / K))
                mapped = mats[op_id] @ base_B[q] + noise
                U_rows.append(mapped[0])
                G_rows.append(mapped[1])
        perm = torch.randperm(inter_size)
        sd[p + "mlp.up_proj.weight"] = torch.stack(U_rows, dim=0)[perm]
        sd[p + "mlp.gate_proj.weight"] = torch.stack(G_rows, dim=0)[perm]
        sd[p + "mlp.down_proj.weight"] = torch.randn(hidden_size, inter_size) * 0.1
    return cfg, sd


snap_dir = find_local_qwen_snapshot("Qwen/Qwen2.5-0.5B")
if snap_dir is not None:
    print(f"Загружаем реальный чекпоинт Qwen2.5-0.5B из: {snap_dir}")
    d16_model, conv_report = QwenToD16Converter.from_pretrained_dir(
        snap_dir, mode=CONVERSION_MODE, device="cpu"
    )
else:
    print("Локальный кэш Qwen2.5-0.5B не найден — используем многослойный стенд с орбитами Z_8 |>< Z_2.")
    mock_cfg, mock_sd = build_synthetic_qwen_with_orbits(num_layers=6, hidden_size=64, inter_size=128)
    d16_model, conv_report = QwenToD16Converter.convert_state_dict(
        mock_sd, mock_cfg, mode=CONVERSION_MODE, device="cpu"
    )

print(f"Конвертация завершена! Сконвертировано слоев: {conv_report.num_layers}")
print(f"Размерность MLP: {conv_report.orig_intermediate_size} вещественных -> {conv_report.d16_intermediate_size} узлов D16Complex")
print(f"Средний косинус орбиты по сети: {conv_report.mean_orbit_score:.4f}")
print(f"Средняя доля 45°-связей Адамара Re(z^2): {conv_report.mean_hadamard_45_pct:.1f}%")

## 2. Таблица связанных нейронов, образующих одно число `D16Complex`
Выведем топ-10 самых чистых орбит выбранного слоя: для каждого узла `D16Complex` показаны 4 исходных вещественных нейрона `n=...`, операция группы $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$ (`op=0..15`) и принадлежность к осевой ветви $\mathrm{Im}(z^2)$ или диагональной $45^\circ$-ветви $\mathrm{Re}(z^2)$.

In [ ]:
INSPECT_LAYER = min(2, conv_report.num_layers - 1)
layer_rep = conv_report.layer_reports[INSPECT_LAYER]
print_orbit_table(layer_rep, top_k=10)

## 3. Круговой 16-секторный компас: как 4 вещественных нейрона раскладываются по секторам `S0..S15`
Нарисуем полярные диаграммы для двух узлов `D16Complex` (`unit_idx=0` и `unit_idx=1`). Жёлтым выделен фундаментальный клин $[0^\circ, 22.5^\circ]$, а цветными стрелками показаны локальные направления всех 4 связанных вещественных нейронов в общей комплексной плоскости $(U_q, G_q)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), subplot_kw={"projection": "polar"})
plot_single_orbit_compass(layer_rep, unit_idx=0, ax=axes[0])
plot_single_orbit_compass(layer_rep, unit_idx=1, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Спектр 16 операций группы $\mathbb{Z}_8 \rtimes \mathbb{Z}_2$ и послойный профиль $45^\circ$-связей Адамара
Слева — распределение всех 16 операций в выбранном слое (синие столбцы `0..7` — осевая подгруппа $\mathrm{Im}(z^2)$, оранжевые столбцы `8..15` — $45^\circ$-повороты и отражения Адамара $\mathrm{Re}(z^2)$). Справа — динамика по всем слоям модели.

In [ ]:
fig_spec = plot_layer_z8z2_spectrum(conv_report, layer_idx=INSPECT_LAYER)
plt.show()

print("Глобальная гистограмма 16 операций Z_8 |>< Z_2 по всем слоям:")
glob_hist = conv_report.global_op_histogram
tot_links = max(int(glob_hist.sum().item()), 1)
for op_id in range(16):
    cnt = int(glob_hist[op_id].item())
    print(f"  {OP_NAMES[op_id]:<34} : {cnt:7,d} ({cnt / tot_links * 100:5.1f}%)")

## 5. Полярные диаграммы 16-секторных коэффициентов `sector_im_gain` и `sector_re_gain`
Посмотрим, какие профили пропускания по 16 секторам аналитически скомпилировались для комплексных узлов в `D16QwenMLP`.

In [ ]:
d16_mlp_layer = d16_model.model.layers[INSPECT_LAYER].mlp
fig_gains = plot_learned_sector_gains(d16_mlp_layer, unit_indices=(0, 1, 2))
plt.show()

## 6. Прогон сконвертированной модели и снятие 18-состояний `D16ComplexTensor`

In [ ]:
d16_model.eval()
d16_model.set_collect_d16_stats(True, reset=True)

sample_ids = torch.randint(0, d16_model.config.vocab_size, (2, 32))
with torch.no_grad():
    out = d16_model(sample_ids)

summary = d16_model.get_d16_state_summary()
counts = summary["state_counts"]
tot = max(summary["total_elements"], 1)

print(f"Выход логитов: {tuple(out.logits.shape)} | Все значения конечны: {bool(torch.isfinite(out.logits).all())}")
print("=" * 72)
print("РАСПРЕДЕЛЕНИЕ АКТИВАЦИЙ СКОНВЕРТИРОВАННОЙ СЕТИ ПО 18 СОСТОЯНИЯМ D16Complex")
print("=" * 72)
for s_idx in range(18):
    c = int(counts[s_idx].item())
    print(f"  {s_idx:2d}: {STATE_LABELS[s_idx]:<14} | {c:9,d} | {c / tot * 100:6.2f}%")